#### Testing generated market level data such that there is no endogeonous spending and spending within geo is uncorrelated across channels

In [ ]:
import numpy as np
import pandas as pd 

from meridian import constants 
from meridian.data import load, test_utils 
from meridian.model import model, spec, prior_distribution 
from meridian.analysis import optimizer, analyzer, visualizer, summarizer, formatter 

import tensorflow as tf 
import tensorflow_probability as tfp

%matplotlib inline
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"Your runtime has {ram_gb:.1f} gigabytes of available RAM\n")

import sys
print(sys.executable)


In [ ]:
from mmm_lab.data_generation.baseline import generate_baseline_geo_data
from mmm_lab.data_generation.marketing import add_marketing_effects, geometric_adstock, hill_saturation


# Set seed for reproducibility
np.random.seed(42)

# Generate baseline data
baseline_df = generate_baseline_geo_data(n_geos=40, n_weeks=104, start_date='2023-01-01')
df = add_marketing_effects(baseline_df, channels=['tv', 'paid_search'])

print(f"\nData shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")

# Store ground truth for validation
ground_truth = df.attrs['channel_params']
print("\n" + "="*80)
print("GROUND TRUTH PARAMETERS (What we're trying to recover)")
print("="*80)
for channel, params in ground_truth.items():
    print(f"\n{channel.upper()}:")
    for k, v in params.items():
        print(f"  {k}: {v}")


GROUND TRUTH MARKETING EFFECTS

TV:
  Total Spend:        $10,490,532
  Total Effect:       11,350,578 bookings
  Overall ROAS:       $1.08
  Avg Saturation:     47.3%
  Adstock Rate:       50.0% (carryover)
  Half-Sat Point (K): $5,000
  Shape (S):          1.0

PAID_SEARCH:
  Total Spend:        $8,502,293
  Total Effect:       19,361,757 bookings
  Overall ROAS:       $2.28
  Avg Saturation:     58.5%
  Adstock Rate:       30.0% (carryover)
  Half-Sat Point (K): $2,000
  Shape (S):          1.5


Data shape: (4160, 15)
Date range: 2023-01-02 00:00:00 to 2024-12-23 00:00:00
Columns: ['date', 'geo', 'region', 'size_tier', 'population', 'baseline_bookings', 'market_conditions', 'demand_perfect', 'demand_good', 'demand_poor', 'spend_tv', 'effect_tv', 'spend_paid_search', 'effect_paid_search', 'total_bookings']

GROUND TRUTH PARAMETERS (What we're trying to recover)

TV:
  weekly_budget: 100000
  budget_variation: 0.2
  geo_budget_variation: 0.15
  adstock_rate: 0.5
  saturation_K: 5000

/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/../add_marketing_effects.py:243: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_saturation = df.groupby('geo').apply(
/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/../add_marketing_effects.py:243: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_saturation = df.groupby('geo').apply(


In [3]:
# Meridian expects specific column structure
# We'll keep it simple for MVP: no media impressions, just spend

meridian_df = df[[
    'date', 'geo', 'region', 'population',
    'total_bookings',  # This will be our KPI
    'spend_tv', 'spend_paid_search',
    # Optional: include for validation later
    'baseline_bookings', 'effect_tv', 'effect_paid_search',
    # Optional: demand controls to test
    'demand_perfect', 'demand_good', 'demand_poor'
]].copy()

print(f"\nMeridian data shape: {meridian_df.shape}")
print(f"\nColumns: {meridian_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(meridian_df.head())

# Save to CSV (Meridian requirement)
csv_path = '../data/simulated_mmm_data.csv'
meridian_df.to_csv(csv_path, index=False)
print(f"\n✓ Data saved to: {csv_path}")


Meridian data shape: (4160, 13)

Columns: ['date', 'geo', 'region', 'population', 'total_bookings', 'spend_tv', 'spend_paid_search', 'baseline_bookings', 'effect_tv', 'effect_paid_search', 'demand_perfect', 'demand_good', 'demand_poor']

First few rows:
        date  geo  region  population  total_bookings     spend_tv  \
0 2023-01-02    0       2     1436350     2550.528702  2636.858641   
1 2023-01-09    0       2     1436350     4136.572120  1938.509521   
2 2023-01-16    0       2     1436350     5220.047312  2549.159023   
3 2023-01-23    0       2     1436350     5198.179926  2371.769610   
4 2023-01-30    0       2     1436350     5949.382675  2629.213833   

   spend_paid_search  baseline_bookings    effect_tv  effect_paid_search  \
0        1200.927101                697  1041.472120          812.056582   
1        1548.950175                705  1469.565144         1962.006976   
2        1503.011845                773  2175.293464         2271.753849   
3        1303.118088

In [4]:
# Map DataFrame columns to Meridian concepts
coord_to_columns = load.CoordToColumns(
    time='date',
    geo='geo',
    population='population',
    kpi='total_bookings',
    
    # For MVP, we'll use spend as proxy for impressions
    # (Meridian prefers impressions but can work with spend)
    media=['spend_tv', 'spend_paid_search'],
    media_spend=['spend_tv', 'spend_paid_search'],
    
    # Optional: controls for demand
#    controls=['demand_good'],  
    controls=[],  # Test no controls first 
)

# Channel naming (Meridian likes clean names)
media_to_channel = {
    'spend_tv': 'TV',
    'spend_paid_search': 'Paid_Search'
}

media_spend_to_channel = {
    'spend_tv': 'TV',
    'spend_paid_search': 'Paid_Search'
}

print("✓ Coordinate mappings defined")

✓ Coordinate mappings defined


In [5]:
# Load data using Meridian's CSV loader
loader = load.CsvDataLoader(
    csv_path=csv_path,
    kpi_type='revenue',  # Treat 'bookings' as revenue for simplicity 
    coord_to_columns=coord_to_columns,
    media_to_channel=media_to_channel,
    media_spend_to_channel=media_spend_to_channel
)

data = loader.load()

/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/.venv/lib/python3.12/site-packages/meridian/data/input_data.py:516: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


In [6]:
def convert_normal_to_lognormal_params(mu, sigma):
    """Convert normal distribution parameters to lognormal distribution parameters.
    
    Args:
        mu (float): Mean of the normal distribution.
        sigma (float): Standard deviation of the normal distribution.
        
    Returns:
        tuple: A tuple containing the mean and standard deviation of the lognormal distribution.
    """
    mean_sq = mu**2
    variance = sigma**2

    sigma_lognormal = np.sqrt(np.log(1+(variance / mean_sq)))
    mu_lognormal = np.log(mean_sq / np.sqrt(mean_sq+variance))
    return mu_lognormal, sigma_lognormal

print(convert_normal_to_lognormal_params(1.0, 1.0))
print(convert_normal_to_lognormal_params(1.5, 0.75))


(np.float64(-0.34657359027997275), np.float64(0.8325546111576977))
(np.float64(0.29389333245105953), np.float64(0.47238072707743883))


In [7]:
# Build media channel arguments: Assume break-even priors for each channel 
custom_priors = {
    'TV': (-0.346, 0.833),
    'Paid_Search': (-0.346, 0.833),
}

roi_m_sigma = list(custom_priors.values())
prior_loc = [first for first, _ in roi_m_sigma]
prior_scale = [second for _, second in roi_m_sigma]

prior = prior_distribution.PriorDistribution(
    roi_m = tfp.distributions.LogNormal(
        loc = prior_loc,
        scale = prior_scale,
        name = constants.ROI_M
    ),
)

model_spec = spec.ModelSpec(
    prior=prior
)

In [8]:
# Initialize model
mmm = model.Meridian(
    input_data=data,
    model_spec=model_spec
)



I0000 00:00:1768800518.871928 10916996 service.cc:148] XLA service 0x16e2aca00 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768800518.872286 10916996 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1768800518.920813 10916996 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [9]:
# Fit the model
import time 
print("Sampling prior...")
start = time.time()
mmm.sample_prior(500)
print(f"Prior sampling took {time.time() - start:.1f} seconds")

start = time.time()
mmm.sample_posterior(n_chains=4, n_adapt=500, n_burnin=500, n_keep=1000)
print(f"Posterior sampling took {time.time() - start:.1f} seconds")

print("\n✓ Model fitting complete!")

Sampling prior...
Prior sampling took 0.1 seconds


2026-01-19 12:28:54.784403: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-19 12:28:55.105367: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:107] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. sanitize_seed/seed
W0000 00:00:1768800535.381658 10916996 assert_op.cc:38] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


Posterior sampling took 631.1 seconds

✓ Model fitting complete!


In [10]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

alt.LayerChart(...)

In [11]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

alt.LayerChart(...)

In [12]:
filename = 'meridian_summary_market_clean_data_control_none.html'
outputpath = 'output/'
mmm_summarizer = summarizer.Summarizer(mmm)
mmm_summarizer.output_model_results_summary(filename=filename, filepath=outputpath)


/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/.venv/lib/python3.12/site-packages/meridian/analysis/analyzer.py:3181: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(
/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/.venv/lib/python3.12/site-packages/meridian/analysis/analyzer.py:928: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(
/Users/jonathan/Desktop/projects/causal_inference/MMM/meridian/.venv/lib/python3.12/site-packages/tensorflow/python/autograph/impl/api.py:371: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  return py_builtins.overload_of(f)(*args)


In [13]:
print("Media,   Prob ROAS >= 0")
list(zip(media_to_channel.values(), 
         (mmm.inference_data.posterior.roi_m >=1).mean(
             dim=('chain', 'draw')).values))

Media,   Prob ROAS >= 0


[('TV', np.float64(0.98525)), ('Paid_Search', np.float64(1.0))]

In [14]:
print("Channel -- Posterior -- Prior")
prior_v_posterior = pd.DataFrame(list(zip(media_to_channel.values(),
                                          (mmm.inference_data.posterior.roi_m).mean(dim=('chain', 'draw')).values,
                                          mmm.inference_data.prior.roi_m.mean(dim=('chain','draw')).values)),
                                          columns = ['media', 'posterior_mean', 'prior_mean'])
print(prior_v_posterior)

Channel -- Posterior -- Prior
         media  posterior_mean  prior_mean
0           TV        1.285881    0.943348
1  Paid_Search        2.116371    1.080148
